In [15]:
import torch
from typing import Optional,List
from d2l import torch as d2l
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [22]:
num_inputs,num_hidden,num_output=784,256,10
epochs=10
batch_size=128
lr=0.1

W1=torch.normal(0,0.01,size=(num_inputs,num_hidden),requires_grad=True)
b1=torch.zeros(num_hidden,requires_grad=True)
W2=torch.normal(0,0.01,size=(num_hidden,num_output),requires_grad=True)
b2=torch.zeros(num_output,requires_grad=True)

params=[W1,b1,W2,b2]


In [26]:
loss=torch.nn.CrossEntropyLoss(reduction="mean")
updater=torch.optim.SGD(params,lr=lr)
def ReLU(X):
    return torch.max(X,torch.zeros_like(X))

def net(X:torch.tensor):
    X=X.reshape((-1,num_inputs))
    h1=ReLU(X@W1+b1)
    return h1@W2+b2

train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)


In [27]:
def train_ch3(epochs,net,loss,updater,train_iter,test_iter):
    for epoch in range(epochs):
        l,a = train_epoch_ch3(net,train_iter,loss,updater)
        print(f"epoch:{epoch} \t loss:{l:.3f} \t acc:{a:.3f}\n")

def accuracy(y_hat,y):
    y_hat = y_hat.argmax(axis=1)
    if len(y_hat) == len(y):
        y_a = y_hat.type(y.dtype)==y
    return float(y_a.type(y.dtype).sum())

def train_epoch_ch3(net,train_iter,loss,updater):
    if isinstance(net,torch.nn.Module):
        net.train()
    metric=Accumulator(3)
    for X,y in train_iter:
        y_hat = net(X)
        l = loss(y_hat,y)
        if isinstance(updater,torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            updater.step()
            metric.add(float(l)*len(y),accuracy(y_hat,y),y.numel())
        else:
            l.sum().backward()
            updater(X.shape[0])
            metric.add(float(l.sum()),accuracy(y_hat,y),y.numel())

    return metric[0]/metric[2],metric[1]/metric[2]
class Accumulator:
    def __init__(self,n):
        self.data = [0.0]*n

    def add(self,*args):
        self.data = [a+float(b) for a,b in zip(self.data,args)]

    def __getitem__(self, idx):
        return self.data[idx]

    def reset(self):
        self.data = [0.0]*len(self.data)


In [28]:
train_ch3(epochs,net,loss,updater,train_iter,test_iter)

C:\Users\wangfengyu\AppData\Local\Temp\ipykernel_33944\555842003.py:23: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  metric.add(float(l)*len(y),accuracy(y_hat,y),y.numel())


epoch:0 	 loss:0.832 	 acc:0.705

epoch:1 	 loss:0.507 	 acc:0.821

epoch:2 	 loss:0.451 	 acc:0.841

epoch:3 	 loss:0.418 	 acc:0.852

epoch:4 	 loss:0.394 	 acc:0.859

epoch:5 	 loss:0.378 	 acc:0.864

epoch:6 	 loss:0.362 	 acc:0.870

epoch:7 	 loss:0.352 	 acc:0.873

epoch:8 	 loss:0.340 	 acc:0.879

epoch:9 	 loss:0.331 	 acc:0.881

